In [ ]:
skfnac_data=pd.read_parquet('benchmark/combined_bench/preds/ready_input_data_with_reg.parquet.gzip')
df = pl.read_parquet(file)
df = df.unique()
df = df.with_columns(pl.col('class').cast(pl.Int64, strict=False))
df = df.filter(pl.col("class").is_in([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13]))
df=df.unique(subset=["class", "lat", "lon"])

suffixes = [
    '_wNDVI', '_mNDVI', '_S', '_A', '_mS', '_mA',
    '_doy_max', '_doy_min', '_start_of_growth', '_end_of_growth',
    '_start_of_decay', '_end_of_decay',
    '_max_growth', '_mean_growth', '_min_growth',
    '_max_decay', '_min_decay', '_mean_decay'
]

indexes=['red', 'nir', 'swir1', 'swir2', 'green', 'blue', 'wrdvi', 'ndre', 'ndyi', 'median_red', 'median_nir', 'median_swir1', 'median_swir2', 'median_blue', 'median_green']
matched_cols = (df.select(cs.ends_with(suffixes)))
matched_cols = (df.select(cs.starts_with(indexes)))

cols_to_select += [*matched_cols.columns]

cols_to_select=  ['sum_t_4', 'sum_t_5', 'sum_t_6', 'sum_t_7', 'sum_t_8', 'sum_t_9', 'sum_t_10', 'sum_prec_4', 'sum_prec_6', 'sum_prec_10', 'median_t_4', 'median_t_6', 'median_t_9', 'median_t_10', 'ndre_S', 'median_red_fitted_8', 'median_nir_fitted_5', 'median_nir_fitted_8', 'median_swir1_fitted_6', 'median_swir1_fitted_7', 'median_swir1_fitted_8', 'median_green_fitted_7', 'median_green_fitted_8', 'median_swir2_fitted_5']

# cols_to_select=[
#     "sum_t_4",
#     "sum_t_5",
#     "sum_t_6",
#     "sum_t_7",
#     "sum_t_8",
#     "sum_t_9",
#     "sum_t_10",
#     "sum_prec_4",
#     "sum_prec_5",
#     "sum_prec_6",
#     "sum_prec_7",
#     "sum_prec_8",
#     "sum_prec_9",
#     "sum_prec_10",
#     "median_t_4",
#     "median_t_5",
#     "median_t_6",
#     "median_t_7",
#     "median_t_8",
#     "median_t_9",
#     "median_t_10",
#     "median_prec_4",
#     "median_prec_5",
#     "median_prec_6",
#     "median_prec_7",
#     "median_prec_8",
#     "median_prec_9",
#     "median_prec_10",
#     "wrdvi_wNDVI",
#     "wrdvi_S",
#     "wrdvi_A",
#     "wrdvi_mS",
#     "wrdvi_mA",
#     "wrdvi_max",
#     "wrdvi_end_of_growth",
#     "ndre_wNDVI",
#     "ndre_S",
#     "ndre_A",
#     "ndre_mS",
#     "ndre_mA",
#     "ndre_max",
#     "ndyi_doy_max",
#     "ndyi_max",
#     "red_min",
#     "red_doy_min",
#     "median_red_fitted_4",
#     "median_red_fitted_5",
#     "median_red_fitted_6",
#     "median_red_fitted_7",
#     "median_red_fitted_8",
#     "median_red_fitted_9",
#     "median_red_fitted_10",
#     "nir_max",
#     "median_nir_fitted_4",
#     "median_nir_fitted_5",
#     "median_nir_fitted_6",
#     "median_nir_fitted_7",
#     "median_nir_fitted_8",
#     "median_nir_fitted_9",
#     "median_nir_fitted_10",
#     "median_blue_fitted_5",
#     "median_blue_fitted_7",
#     "median_blue_fitted_8",
#     "median_blue_fitted_9",
#     "median_swir1_fitted_5",
#     "median_swir1_fitted_6",
#     "median_swir1_fitted_7",
#     "median_swir1_fitted_8",
#     "median_swir1_fitted_9",
#     "median_green_fitted_5",
#     "median_green_fitted_6",
#     "median_green_fitted_7",
#     "median_green_fitted_8",
#     "median_green_fitted_9",
#     "swir2_min",
#     "median_swir2_fitted_4",
#     "median_swir2_fitted_5",
#     "median_swir2_fitted_6",
#     "median_swir2_fitted_7",
#     "median_swir2_fitted_8",
#     "median_swir2_fitted_9",
#     "median_swir2_fitted_10"
# ]

sample = df.drop_nulls(subset=cols_to_select)
temp_df = sample.to_pandas()

model = CatBoostClassifier()
model.load_model(f'models/model_for_using.cbm')

# Рассчёт весов классов
class_counts = Counter(valid_data['class'])
total = sum(class_counts.values())
class_weights = {k: np.log(total / v) for k, v in class_counts.items()}
valid_sample_weights = np.array([class_weights[label] for label in valid_data['class']])

# Стратификация по комбинации рег+класс (если ожидается такая диспропорция). Можно оставить только по классу.
valid_data['stratify_key'] = valid_data['reg'].astype(str) + '_' + valid_data['class'].astype(str)

# Трен/тест сплит
(val_pred_train, val_pred_test,
 val_class_train, val_class_test,
 val_sw_train, val_sw_test) = train_test_split(
    valid_data,
    valid_data['class'],
    valid_sample_weights,
    test_size=0.2,
    random_state=42,
    stratify=valid_data['stratify_key']
)

# ОБЕСПЕЧИВАЕМ НАЛИЧИЕ ВСЕХ КЛАССОВ (фиктивными строками)
missing_classes = set(cat_variable) - set(val_class_train.unique())
for cls in missing_classes:
    # Берём первую строку temp_df с этим классом — важно temp_df должен содержать такие строки!
    fake_source = temp_df.loc[temp_df['class'] == cls]
    if fake_source.empty:
        raise ValueError(f'Нет ни одной строки для фиктивного класса {cls} в temp_df')
    fake_row = fake_source.iloc[0][columns].to_dict()
    fake_row['class'] = cls
    val_pred_train = pd.concat([val_pred_train, pd.DataFrame([fake_row])], ignore_index=True)
    val_class_train = np.append(val_class_train, cls)  # корректно добавляем в массив классов
    val_sw_train = np.append(val_sw_train, 1e-8)       # минимальный вес

# (sample_weight нужен именно для train, сам train-еще не обновили sample_weight!)
sample_weight = val_sw_train

model_finetune = CatBoostClassifier(iterations=100, learning_rate=0.01)
model_finetune.fit(
    val_pred_train[columns],
    val_class_train,
    sample_weight=sample_weight,
    init_model=model,
    verbose=200,
)

# model_finetune.save_model(f'models/model_for_using.cbm')